# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

## Implementación para SciFact

Ejecuta las celdas en orden. El corpus oficial de BEIR incluye documentos, consultas y qrels; aquí se evalúa la partición `test`. La descarga requiere acceso a internet.


In [ ]:
# Dependencias
%pip install -q requests pandas numpy scikit-learn nltk fastapi uvicorn


### 1. Descargar y cargar SciFact

SciFact está en inglés. El preprocesamiento usa minúsculas, tokenización por expresiones regulares, eliminación de stopwords inglesas y stemming Porter. Se conservan los IDs originales y cada abstract completo es un documento.


In [1]:
from pathlib import Path
import json, re, zipfile, requests
import pandas as pd

DATA_DIR = Path("data/scifact")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / "scifact.zip"
URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"
if not ZIP_PATH.exists():
    response = requests.get(URL, timeout=120)
    response.raise_for_status()
    ZIP_PATH.write_bytes(response.content)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(DATA_DIR)

ROOT = DATA_DIR / "scifact"
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

corpus_rows = read_jsonl(ROOT / "corpus.jsonl")
query_rows = read_jsonl(ROOT / "queries.jsonl")
corpus = {str(d["_id"]): {"title": d.get("title", ""), "text": d.get("text", "")} for d in corpus_rows}
queries = {str(q["_id"]): q["text"] for q in query_rows}
qrels = pd.read_csv(ROOT / "qrels" / "test.tsv", sep="\t", dtype={"query-id":str,"corpus-id":str})
qrels.columns = [c.lower() for c in qrels.columns]
qrels["query-id"] = qrels["query-id"].astype(str)
qrels["corpus-id"] = qrels["corpus-id"].astype(str)
print(f"Documentos: {len(corpus):,} | Consultas: {len(queries):,} | Juicios test: {len(qrels):,}")
print("Ejemplo ID original:", next(iter(corpus)))


Documentos: 5,183 | Consultas: 1,109 | Juicios test: 339
Ejemplo ID original: 4983


### 2. Preprocesamiento e índice invertido

Se concatenan título y texto para indexar, y se mantiene separado el título para mostrarlo en resultados. El índice invertido guarda frecuencia por término y documento.


In [2]:
from collections import Counter, defaultdict
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import math

stemmer = PorterStemmer()
STOPWORDS = set(ENGLISH_STOP_WORDS)
def tokenize(text):
    words = re.findall(r"[a-z]+", (text or "").lower())
    return [stemmer.stem(w) for w in words if w not in STOPWORDS]

doc_ids = list(corpus)
doc_tokens = {}
inverted_index = defaultdict(dict)
doc_lengths = {}
for doc_id, doc in corpus.items():
    tokens = tokenize(doc["title"] + " " + doc["text"])
    freqs = Counter(tokens)
    doc_tokens[doc_id] = freqs
    doc_lengths[doc_id] = sum(freqs.values())
    for term, freq in freqs.items():
        inverted_index[term][doc_id] = freq
N = len(doc_ids)
avgdl = sum(doc_lengths.values()) / max(N, 1)
vocabulary = set(inverted_index)

# Normas completas de los vectores TF-IDF, precalculadas una sola vez.
doc_tfidf_norms = {}
for doc_id, freqs in doc_tokens.items():
    norm_sq = 0.0
    for term, tf in freqs.items():
        df = len(inverted_index[term])
        idf = math.log((N + 1) / (df + 1)) + 1
        weight = (1 + math.log(tf)) * idf
        norm_sq += weight * weight
    doc_tfidf_norms[doc_id] = math.sqrt(norm_sq)
print(f"Vocabulario: {len(vocabulary):,} términos | Promedio: {avgdl:.1f} términos/documento | Normas TF-IDF precalculadas")


Vocabulario: 20,781 términos | Promedio: 136.2 términos/documento | Normas TF-IDF precalculadas


### 3. Modelos de recuperación

TF-IDF usa IDF suavizado y similitud coseno. BM25 usa `k1=1.5` y `b=0.75`. Ambos devuelven IDs únicos ordenados por puntuación descendente.


In [3]:
def search_tfidf(query, top_k=10):
    qfreq = Counter(tokenize(query))
    if not qfreq or top_k <= 0:
        return []
    qweights = {term: qf * (math.log((N + 1) / (len(inverted_index.get(term, {})) + 1)) + 1) for term, qf in qfreq.items()}
    qnorm = math.sqrt(sum(weight * weight for weight in qweights.values())) or 1.0
    scores = defaultdict(float)
    for term, qweight in qweights.items():
        idf = math.log((N + 1) / (len(inverted_index.get(term, {})) + 1)) + 1
        for doc_id, tf in inverted_index.get(term, {}).items():
            doc_weight = (1 + math.log(tf)) * idf
            scores[doc_id] += doc_weight * qweight
    ranked = [(doc_id, score / ((doc_tfidf_norms[doc_id] or 1.0) * qnorm)) for doc_id, score in scores.items() if score > 0]
    return sorted(ranked, key=lambda item: item[1], reverse=True)[:top_k]

def search_bm25(query, top_k=10, k1=1.5, b=0.75):
    qfreq = Counter(tokenize(query))
    scores = defaultdict(float)
    for term, qf in qfreq.items():
        postings = inverted_index.get(term, {})
        df = len(postings)
        if not df:
            continue
        idf = math.log(1 + (N - df + 0.5) / (df + 0.5))
        for doc_id, tf in postings.items():
            denom = tf + k1 * (1 - b + b * doc_lengths[doc_id] / (avgdl or 1))
            scores[doc_id] += idf * (tf * (k1 + 1) / denom) * qf
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:max(0, top_k)]

print("BM25:", search_bm25("Does vitamin D prevent cancer?", 3))
print("TF-IDF:", search_tfidf("Does vitamin D prevent cancer?", 3))


BM25: [('12074066', 21.019841721445186), ('23267371', 19.26135678452331), ('38551172', 18.60864190826309)]
TF-IDF: [('22843838', 0.3279443411042974), ('30720103', 0.3251876393464216), ('23267371', 0.3187765064075655)]


### 4. Evaluación reproducible en test

Precision@10 divide los relevantes recuperados entre 10; Recall@10 divide entre el total de relevantes del qrel; NDCG@10 usa ganancias graduadas. Las consultas sin juicios de relevancia no se promedian.


In [4]:
from math import log2

qrels_by_query = {qid: dict(zip(g["corpus-id"], g["score"].astype(int))) for qid, g in qrels.groupby("query-id")}
test_query_ids = [qid for qid in queries if qid in qrels_by_query]

def metrics_at_k(ranked_ids, relevant, k=10):
    top = ranked_ids[:k]
    rel = [relevant.get(did, 0) for did in top]
    p = sum(x > 0 for x in rel) / k
    r = sum(x > 0 for x in rel) / max(1, sum(v > 0 for v in relevant.values()))
    dcg = sum((2**x - 1) / log2(i + 2) for i, x in enumerate(rel))
    ideal = sorted(relevant.values(), reverse=True)[:k]
    idcg = sum((2**x - 1) / log2(i + 2) for i, x in enumerate(ideal))
    return p, r, dcg / idcg if idcg else 0.0

def evaluate(search_fn, k=10):
    rows=[]
    for qid in test_query_ids:
        ranked = [did for did, _ in search_fn(queries[qid], k)]
        rows.append(metrics_at_k(ranked, qrels_by_query[qid], k))
    return pd.DataFrame(rows, columns=[f"Precision@{k}",f"Recall@{k}",f"NDCG@{k}"]).mean()

results = pd.DataFrame({"TF-IDF": evaluate(search_tfidf), "BM25": evaluate(search_bm25)}).T
display(results)


        Precision@10  Recall@10   NDCG@10
TF-IDF      0.090333   0.821833  0.664642
BM25        0.091333   0.829500  0.693210


### 5. Comparación e interpretación

La tabla anterior presenta el promedio macro sobre las consultas de `test` que tienen qrels. Compara NDCG para valorar el orden y Precision/Recall para valorar cuántos relevantes aparecen en los diez primeros. El mejor modelo depende de estos resultados; no se debe concluir sin ejecutar la evaluación.


In [5]:
display(results.round(4))
print("Consultas evaluadas:", len(test_query_ids))
print("Mejor NDCG@10:", results["NDCG@10"].idxmax())


        Precision@10  Recall@10  NDCG@10
TF-IDF        0.0903     0.8218   0.6646
BM25          0.0913     0.8295   0.6932
Consultas evaluadas: 300
Mejor NDCG@10: BM25


### 6. Servicio web FastAPI

Esta celda crea `app.py`. Ejecuta `uvicorn app:app --host 0.0.0.0 --port 8000` en el entorno donde también esté disponible `data/scifact/scifact/`. La publicación pública se realiza desplegando ese entorno en un proveedor compatible con Python y exponiendo el puerto 8000.


In [6]:
from pathlib import Path
app_source = '"""SciFact retrieval API: TF-IDF cosine and BM25."""\nfrom collections import Counter, defaultdict\nimport json\nimport math\nimport re\nimport zipfile\nfrom pathlib import Path\n\nimport requests\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\nfrom fastapi.responses import HTMLResponse\nfrom nltk.stem import PorterStemmer\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS\n\nDATA_DIR = Path("data/scifact")\nROOT = DATA_DIR / "scifact"\nDATA_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"\n\n\ndef ensure_data():\n    if (ROOT / "corpus.jsonl").exists():\n        return\n    DATA_DIR.mkdir(parents=True, exist_ok=True)\n    archive = DATA_DIR / "scifact.zip"\n    if not archive.exists():\n        response = requests.get(DATA_URL, timeout=180)\n        response.raise_for_status()\n        archive.write_bytes(response.content)\n    with zipfile.ZipFile(archive) as zf:\n        zf.extractall(DATA_DIR)\n\n\nensure_data()\nwith (ROOT / "corpus.jsonl").open(encoding="utf-8") as f:\n    documents = {str(d["_id"]): d for d in (json.loads(line) for line in f)}\n\nstemmer = PorterStemmer()\nstopwords = set(ENGLISH_STOP_WORDS)\n\n\ndef tokenize(text):\n    return [stemmer.stem(w) for w in re.findall(r"[a-z]+", (text or "").lower()) if w not in stopwords]\n\n\nindex = defaultdict(dict)\ndoc_lengths = {}\ndoc_frequencies = {}\nfor doc_id, doc in documents.items():\n    frequencies = Counter(tokenize(doc.get("title", "") + " " + doc.get("text", "")))\n    doc_frequencies[doc_id] = frequencies\n    doc_lengths[doc_id] = sum(frequencies.values())\n    for term, frequency in frequencies.items():\n        index[term][doc_id] = frequency\n\nN = len(documents)\navgdl = sum(doc_lengths.values()) / max(1, N)\ndoc_tfidf_norms = {}\nfor doc_id, frequencies in doc_frequencies.items():\n    norm_sq = 0.0\n    for term, tf in frequencies.items():\n        df = len(index[term])\n        idf = math.log((N + 1) / (df + 1)) + 1\n        weight = (1 + math.log(tf)) * idf\n        norm_sq += weight * weight\n    doc_tfidf_norms[doc_id] = math.sqrt(norm_sq)\napp = FastAPI(title="SciFact Retrieval API", version="1.0.0")\n\n\nclass SearchRequest(BaseModel):\n    query: str = Field(min_length=1)\n    model: str = "bm25"\n    top_k: int = Field(default=10, ge=1, le=100)\n\n\n@app.get("/", response_class=HTMLResponse)\ndef root():\n    page = r\'\'\'<!doctype html>\n<html lang="es">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1">\n  <title>SciFact | Buscador</title>\n  <style>\n    :root{color-scheme:light;--ink:#172033;--muted:#667085;--line:#dfe5ed;--blue:#2556c7;--bg:#f5f7fb}\n    *{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:16px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif}\n    main{max-width:900px;margin:40px auto;padding:0 20px}.card{background:white;border:1px solid var(--line);border-radius:16px;padding:26px;box-shadow:0 8px 24px #1b2b4a0a}\n    h1{margin:0 0 6px;font-size:28px}p{color:var(--muted);margin:0 0 20px}.meta{font-size:14px;color:var(--muted)}\n    form{display:grid;grid-template-columns:1fr 180px 110px auto;gap:12px;align-items:end;margin-top:22px}\n    label{display:grid;gap:6px;font-size:14px;font-weight:600}input,select,button{font:inherit;border-radius:9px;padding:10px 12px;border:1px solid var(--line);min-height:44px;background:#fff;color:var(--ink)}\n    button{background:var(--blue);color:white;border-color:var(--blue);font-weight:650;cursor:pointer}button:disabled{opacity:.65;cursor:wait}\n    #status{margin:18px 0 8px;color:var(--muted);min-height:24px}.result{border-top:1px solid var(--line);padding:16px 0}.result h3{font-size:17px;margin:0 0 5px}.result p{margin:6px 0;color:#344054}.tag{font-size:12px;color:var(--muted)}\n    .links{display:flex;gap:16px;margin-top:18px;font-size:14px}.links a{color:var(--blue)}\n    @media(max-width:700px){main{margin:20px auto}form{grid-template-columns:1fr 1fr}.query{grid-column:1/-1}button{grid-column:1/-1}}\n  </style>\n</head>\n<body>\n<main>\n  <section class="card">\n    <h1>Buscador SciFact</h1>\n    <p>Recuperación de artículos científicos por consulta. Prueba BM25 y TF‑IDF sobre el corpus SciFact.</p>\n    <div class="meta"><span id="health">Comprobando servicio…</span> · Corpus: __DOC_COUNT__ documentos</div>\n    <form id="search-form">\n      <label class="query">Consulta<input id="query" name="query" required placeholder="Ej.: Vitamin D prevents cancer"></label>\n      <label>Modelo<select id="model" name="model"><option value="bm25">BM25</option><option value="tfidf">TF‑IDF (coseno)</option></select></label>\n      <label>Resultados<input id="top-k" name="top_k" type="number" min="1" max="100" value="10" required></label>\n      <button id="submit" type="submit">Buscar</button>\n    </form>\n    <div id="status" role="status" aria-live="polite"></div>\n    <section id="results" aria-label="Resultados de búsqueda"></section>\n    <div class="links"><a href="/docs">API interactiva</a><a href="/health">Estado del servicio</a></div>\n  </section>\n</main>\n<script>\nconst form=document.getElementById(\'search-form\'),statusBox=document.getElementById(\'status\'),resultsBox=document.getElementById(\'results\'),submitButton=document.getElementById(\'submit\');\nfetch(\'/health\').then(r=>r.json()).then(d=>{document.getElementById(\'health\').textContent=`Servicio activo · ${d.documents} documentos`}).catch(()=>{document.getElementById(\'health\').textContent=\'No se pudo comprobar el estado del servicio\'});\nform.addEventListener(\'submit\',async event=>{\n  event.preventDefault();resultsBox.replaceChildren();submitButton.disabled=true;statusBox.textContent=\'Buscando…\';\n  const payload={query:document.getElementById(\'query\').value.trim(),model:document.getElementById(\'model\').value,top_k:Number(document.getElementById(\'top-k\').value)};\n  try{\n    const response=await fetch(\'/search\',{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify(payload)});\n    const data=await response.json();if(!response.ok)throw new Error(data.detail||\'Error en la búsqueda\');\n    statusBox.textContent=`${data.results.length} resultados · ${data.model.toUpperCase()}`;\n    if(!data.results.length){const empty=document.createElement(\'p\');empty.textContent=\'No se encontraron documentos para esta consulta.\';resultsBox.append(empty);return;}\n    for(const item of data.results){\n      const article=document.createElement(\'article\');article.className=\'result\';\n      const heading=document.createElement(\'h3\');heading.textContent=`${item.rank}. ${item.title||\'Documento sin título\'}`;\n      const meta=document.createElement(\'div\');meta.className=\'tag\';meta.textContent=`ID: ${item.document_id} · Puntuación: ${Number(item.score).toFixed(4)}`;\n      const snippet=document.createElement(\'p\');snippet.textContent=item.snippet;\n      article.append(heading,meta,snippet);resultsBox.append(article);\n    }\n  }catch(error){statusBox.textContent=`No se pudo completar la búsqueda: ${error.message}`;}\n  finally{submitButton.disabled=false;}\n});\n</script>\n</body>\n</html>\'\'\'.replace("__DOC_COUNT__", str(N))\n    return HTMLResponse(content=page)\n\n\ndef retrieve(query, model, top_k):\n    qfreq = Counter(tokenize(query))\n    scores = defaultdict(float)\n    if model == "bm25":\n        for term, qf in qfreq.items():\n            postings = index.get(term, {})\n            df = len(postings)\n            if not df:\n                continue\n            idf = math.log(1 + (N - df + 0.5) / (df + 0.5))\n            for doc_id, tf in postings.items():\n                denominator = tf + 1.5 * (0.25 + 0.75 * doc_lengths[doc_id] / (avgdl or 1))\n                scores[doc_id] += idf * (tf * 2.5 / denominator) * qf\n    else:\n        qweights = {t: f * (math.log((N + 1) / (len(index.get(t, {})) + 1)) + 1) for t, f in qfreq.items()}\n        qnorm = math.sqrt(sum(v * v for v in qweights.values())) or 1.0\n        for term, qweight in qweights.items():\n            idf = math.log((N + 1) / (len(index.get(term, {})) + 1)) + 1\n            for doc_id, tf in index.get(term, {}).items():\n                weight = (1 + math.log(tf)) * idf\n                scores[doc_id] += weight * qweight\n        for doc_id in list(scores):\n            scores[doc_id] /= (doc_tfidf_norms[doc_id] or 1.0) * qnorm\n    return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]\n\n\n@app.get("/health")\ndef health():\n    return {"status": "ok", "corpus": "SciFact", "documents": N}\n\n\n@app.post("/search")\ndef search(request: SearchRequest):\n    model = request.model.lower()\n    if model not in {"bm25", "tfidf"}:\n        raise HTTPException(status_code=400, detail="model debe ser bm25 o tfidf")\n    results = []\n    for rank, (doc_id, score) in enumerate(retrieve(request.query, model, request.top_k), 1):\n        doc = documents[doc_id]\n        results.append({\n            "rank": rank,\n            "document_id": doc_id,\n            "score": float(score),\n            "title": doc.get("title", ""),\n            "snippet": doc.get("text", "")[:300],\n        })\n    return {"model": model, "results": results}\n'
Path("app.py").write_text(app_source, encoding="utf-8")
print("Creado app.py")


Creado app.py


Prueba local tras crear `app.py` (en una terminal):

```bash
uvicorn app:app --host 0.0.0.0 --port 8000
```

Luego consulta `GET /health` y `POST /search` con JSON como `{"query":"vitamin D cancer","model":"bm25","top_k":5}`. Servicio desplegado en Render (plan gratuito): **URL pública: https://scifact-retrieval-api.onrender.com**. Documentación interactiva: https://scifact-retrieval-api.onrender.com/docs. Prueba primero `GET /health`; Render puede dormir la instancia tras inactividad y el primer acceso puede tardar cerca de un minuto.
